# 02 - Launch and stop (guarded)

Rents a GPU server, waits for it, shows how to connect and **always stops it at the end**.

> **This notebook spends money only if you set `CONFIRM_RENT = True`.** With the default `False` it just prints the quote.

You need an API key in the environment: `export GPUNZ_API_KEY=gpz_xxx` (create one in the gpu.nz console or run `gpunz login`).

In [ ]:
import sys
sys.path.insert(0, "../python")

from gpunz_client import GpunzClient, connection_info, format_money

# ---- safety switch ---------------------------------------------------------
CONFIRM_RENT = False      # set to True to really rent (billed per minute)
TEMPLATE = "pytorch"      # try: comfyui, ollama, vllm (see client.list_templates())
MIN_VRAM_GB = 16
DISK_GB = 50
MAX_PRICE_RUB_H = 40      # never accept an offer above this price per hour
GPUS = "rtx-4060-ti-16gb,rtx-5060-ti-16gb,rtx-a4000-16gb,rtx-3090-24gb,rtx-4090-24gb"
# -----------------------------------------------------------------------------

client = GpunzClient()

In [ ]:
offers = client.search_offers(
    gpu=GPUS, num_gpus=1, min_vram_gb=MIN_VRAM_GB, min_disk_gb=DISK_GB, min_reliability=0.95,
    max_price_kop_h=int(MAX_PRICE_RUB_H * 100), order="price", limit=3,
)
if not offers:
    raise SystemExit("No offer under the price cap. Raise MAX_PRICE_RUB_H or lower MIN_VRAM_GB.")
offer = offers[0]
kop_h = offer["price"]["on_demand_kop_h"]
print(f"Offer {offer['id']}: {offer['gpu']['name']} x{offer['gpu']['count']}, {offer['gpu']['vram_gb']} GB")
print(f"Price: {format_money(kop_h)}/h = {format_money(kop_h / 60)}/min")
print("RENT ENABLED" if CONFIRM_RENT else "Dry run: set CONFIRM_RENT = True to rent.")

In [ ]:
instance = None
if CONFIRM_RENT:
    instance = client.create_instance(offer["id"], TEMPLATE, DISK_GB, name="notebook-example")
    print("Created", instance["id"], "- billing has started")
else:
    print("Skipped: CONFIRM_RENT is False")

In [ ]:
if instance:
    try:
        instance = client.wait_until_running(
            instance["id"], timeout_s=20 * 60, on_update=lambda i: print("status:", i["status"])
        )
        info = connection_info(instance)  # the SSH password is not included
        print("SSH:", info["ssh_command"])
        for u in info["urls"]:
            print(f"URL [{u['label']}]:", u["url"])
    except BaseException as exc:  # includes KeyboardInterrupt: never leave a server running by accident
        print("Not ready:", repr(exc))
        print("Run the next cell to stop/destroy the server.")

## Cleanup

Run this when you are done. `stop` keeps the disk (small disk-only fee); `destroy` deletes everything and ends all billing.

In [ ]:
STOP_ACTION = "destroy"   # "stop" keeps the disk, "destroy" deletes it irreversibly

if instance:
    if STOP_ACTION == "stop":
        print(client.stop_instance(instance["id"])["status"])
    else:
        print(client.destroy_instance(instance["id"])["status"])
else:
    print("Nothing to clean up.")